In [ ]:
import copy
import shutil
import os
import glob
import time
import sys
from IPython.display import display, Image, clear_output

from big_deep import load_recent_model
from action_manager import ActionManager
from shared_mcts import MCTS
from setup_game import setup_game
from action_phase import Phase, get_action_str
from dice import roll_dice

def show_visual():
    list_of_files = glob.glob('game_visuals/*.png')

    if list_of_files:
        latest_file = max(list_of_files, key=os.path.getctime)
        
        # Optional: Clears the previous image before showing the new one
        # This creates a nice frame-by-frame animation effect instead of a long scroll
        clear_output(wait=True) 
        
        # 3. Display the image inline in the notebook
        display(Image(filename=latest_file))

def main():
    # --- HARDCODED SETUP ---
    HUMAN_PLAYER = 1           # 1 for First Player, -1 for Second Player
    MCTS_ITERATIONS = 100      # Number of search iterations for the model's turn
    # -----------------------

    print("Loading Alpharmada Model...")
    model, _ = load_recent_model()
    model.eval()
    model.compile_fast_policy()
    
    action_manager = ActionManager()
    
    print("Setting up randomized game...")
    shutil.rmtree("game_visuals", ignore_errors=True)
    os.makedirs("game_visuals", exist_ok=True)
    game = setup_game()
    
    print("Initializing Shared Tree MCTS...")
    mcts = MCTS(copy.deepcopy(game), action_manager, model)
    print(f"Game Start! You are Player {'1 (First)' if HUMAN_PLAYER == 1 else '2 (Second)'}.")

    game.debuging_visual = True
    game.visualize("Game Start!")
    show_visual()

    # --- MAIN GAME LOOP ---
    while game.winner == 0.0:
        

        # 1. Chance Node (Automatic Dice Roll)
        if game.phase == Phase.ATTACK_ROLL_DICE:
            dice_roll = roll_dice(game.attack_info.dice_to_roll)
            action = ('roll_dice_action', dice_roll)
            
            print(f"[DICE] {get_action_str(game, action)}")

        else:
            valid_actions = game.get_valid_actions()
            if len(valid_actions) == 1 :
                time.sleep(2)
                action = valid_actions[0]

            # 2. Human Turn
            elif game.decision_player == HUMAN_PLAYER:
                print(f"--- YOUR TURN (Phase: {game.phase.name}) ---")
                
                # Display actions
                for i, action in enumerate(valid_actions, start=1):
                    print(f"  [{i}] {get_action_str(game, action)}")
                    
                # Input loop
                while True:
                    print(flush=True)
                    sys.stdout.flush()
                    
                    # 2. Give the VS Code UI a fraction of a second to catch up and focus
                    time.sleep(0.1)
                    user_input = input(f"Select an action [1-{len(valid_actions)}]: ")
                    
                    # Guard against Jupyter/VS Code NoneType or ghost empty strings
                    if user_input is None or not str(user_input).strip():
                        continue 
                        
                    user_input = str(user_input).strip()
                    
                    try:
                        action_index = int(user_input) - 1
                        
                        if 0 <= action_index < len(valid_actions):
                            action = valid_actions[action_index]
                            confirm = input(f" You chose: {get_action_str(game, action)}. (Y/n, or press Enter): ")
                            
                            # Handle potential None returns in notebook environments
                            if confirm is None:
                                confirm = ""
                            else:
                                confirm = str(confirm).strip().lower()
                            
                            # Broaden the acceptance criteria
                            if confirm in ['', 'y', 'yes']:
                                break
                            elif confirm in ['n', 'no']:
                                print("Selection cancelled. Let's try again.")
                            else:
                                # repr() will expose any hidden characters causing the failure
                                print(f"Invalid confirmation input (received: {repr(confirm)}). Let's try again.")
                                
                        else:
                            print(f"Invalid index. Please enter a number between 1 and {len(valid_actions)}.")
                            
                    except ValueError:
                        print("Invalid input. Please enter a valid integer.")


            # 3. Model Turn
            else:
                print(f"\n--- MODEL'S TURN (Phase: {game.phase.name}) ---")
                print(f"Thinking for {MCTS_ITERATIONS} iterations...")
                
                # Run parallel workers on the shared tree
                mcts.shared_search(iteration=MCTS_ITERATIONS)
                
                action = mcts.get_best_action()
                print(f"> Model chose: {get_action_str(game, action)}")
            
        game.apply_action(action)
        mcts.advance_tree(action, game.get_snapshot())
        
        show_visual()


    # --- GAME OVER ---
    print("GAME OVER")
    if (game.winner>0) == (HUMAN_PLAYER>0):
        print("Congratulations, you won!")
    else:
        print("The model defeated you.")

if __name__ == "__main__":
    main()